In [ ]:

# risk_events_analysis.py
# Author: Victor Mpofu (Senior MI Analyst)
# Purpose: Statistical analysis pipeline to manage risk events of clients

import os
import math
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import PoissonRegressor, LogisticRegression
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_fscore_support,
    mean_absolute_error
)

warnings.filterwarnings("ignore")
sns.set(style="whitegrid", context="talk")


# Configuration


@dataclass
class RiskConfig:
    date_col: str = "event_date"
    resolve_col: str = "resolved_date"
    client_col: str = "client_id"
    event_id_col: str = "event_id"
    event_type_col: str = "event_type"
    severity_col: str = "severity_score"   # numeric (e.g., 1-5 or continuous)
    loss_col: str = "monetary_loss"        # numeric currency
    status_col: str = "status"             # e.g., 'Open', 'Closed'
    escalated_col: str = "escalated"       # 0/1 or True/False
    control_failed_col: str = "control_failed"  # 0/1 or True/False

    # Severity thresholds for high severity classification:
    high_severity_threshold: float = 4.0   # adjust for your scale (e.g., >=4 considered high)

    # Minimum records to include client in modeling (avoid overfitting/extreme sparsity)
    min_events_per_client: int = 5

    # Output paths
    output_dir: str = "./outputs"
    report_xlsx: str = "risk_report.xlsx"
    plots_dir: str = "plots"


# Data Load & Cleaning


def load_data(input_path: str) -> pd.DataFrame:
    ext = os.path.splitext(input_path)[1].lower()
    if ext == ".csv":
        df = pd.read_csv(input_path)
    elif ext == ".parquet":
        df = pd.read_parquet(input_path)
    elif ext in [".xlsx", ".xls"]:
        df = pd.read_excel(input_path, engine="openpyxl")
    else:
        raise ValueError(f"Unsupported file type: {ext}")
    return df

def clean_data(df: pd.DataFrame, cfg: RiskConfig) -> pd.DataFrame:
    # Standardize column names (strip/lowcase)
    df.columns = [c.strip() for c in df.columns]

    # Parse dates
    df[cfg.date_col] = pd.to_datetime(df[cfg.date_col], errors="coerce")
    if cfg.resolve_col in df.columns:
        df[cfg.resolve_col] = pd.to_datetime(df[cfg.resolve_col], errors="coerce")

    # Coerce numerics
    for col in [cfg.severity_col, cfg.loss_col]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Booleans to int (0/1)
    for col in [cfg.escalated_col, cfg.control_failed_col]:
        if col in df.columns:
            df[col] = df[col].astype(str).str.lower().replace(
                {"true": 1, "false": 0, "yes": 1, "no": 0}
            ).astype(float)  # float for modeling compatibility

    # Remove obvious bad rows
    df = df.dropna(subset=[cfg.client_col, cfg.date_col])
    # Ensure loss non-negative
    if cfg.loss_col in df.columns:
        df[cfg.loss_col] = df[cfg.loss_col].clip(lower=0)

    # Deduplicate by event_id if present
    if cfg.event_id_col in df.columns:
        df = df.sort_values(by=[cfg.event_id_col, cfg.date_col]).drop_duplicates(
            subset=[cfg.event_id_col], keep="last"
        )

    # Derive helper fields
    df["event_month"] = df[cfg.date_col].dt.to_period("M").astype(str)

    # Time-to-resolution (days)
    if cfg.resolve_col in df.columns:
        df["ttr_days"] = (df[cfg.resolve_col] - df[cfg.date_col]).dt.days
        # If status is closed but resolved_date is missing, impute with median TTR of similar events
        if cfg.status_col in df.columns:
            closed_mask = (df[cfg.status_col].str.lower() == "closed") & df["ttr_days"].isna()
            median_ttr = df.loc[~df["ttr_days"].isna(), "ttr_days"].median()
            df.loc[closed_mask, "ttr_days"] = median_ttr

    return df

# Key Risk Indicators (KRIs)


def compute_kri(df: pd.DataFrame, cfg: RiskConfig) -> Dict[str, float]:
    # Overall KRIs
    total_events = len(df)
    unique_clients = df[cfg.client_col].nunique()
    avg_events_per_client = total_events / max(unique_clients, 1)

    # Severity metrics
    sev_mean = df[cfg.severity_col].mean() if cfg.severity_col in df.columns else np.nan
    sev_p90 = df[cfg.severity_col].quantile(0.90) if cfg.severity_col in df.columns else np.nan
    high_sev_rate = (
        (df[cfg.severity_col] >= cfg.high_severity_threshold).mean()
        if cfg.severity_col in df.columns
        else np.nan
    )

    # Loss metrics
    total_loss = df[cfg.loss_col].sum() if cfg.loss_col in df.columns else np.nan
    loss_p95 = df[cfg.loss_col].quantile(0.95) if cfg.loss_col in df.columns else np.nan
    loss_mean = df[cfg.loss_col].mean() if cfg.loss_col in df.columns else np.nan

    # TTR metrics
    mttr = df["ttr_days"].mean() if "ttr_days" in df.columns else np.nan
    ttr_p90 = df["ttr_days"].quantile(0.90) if "ttr_days" in df.columns else np.nan

    # Recurrence (client has >1 events)
    recurrence_rate = (
        (df.groupby(cfg.client_col).size() > 1).mean()
    )

    # Escalation rate
    escalation_rate = df[cfg.escalated_col].mean() if cfg.escalated_col in df.columns else np.nan

    # Control failure rate
    control_failure_rate = df[cfg.control_failed_col].mean() if cfg.control_failed_col in df.columns else np.nan

    return {
        "total_events": total_events,
        "unique_clients": unique_clients,
        "avg_events_per_client": avg_events_per_client,
        "severity_mean": sev_mean,
        "severity_p90": sev_p90,
        "high_severity_rate": high_sev_rate,
        "total_loss": total_loss,
        "loss_mean": loss_mean,
        "loss_p95": loss_p95,
        "mttr_days_mean": mttr,
        "mttr_days_p90": ttr_p90,
        "recurrence_rate_clients": recurrence_rate,
        "escalation_rate_events": escalation_rate,
        "control_failure_rate_events": control_failure_rate,
    }

def client_level_metrics(df: pd.DataFrame, cfg: RiskConfig) -> pd.DataFrame:
    # Aggregations per client
    aggs = {
        cfg.event_id_col: "count" if cfg.event_id_col in df.columns else ("size"),
        cfg.severity_col: ["mean", "max"],
        cfg.loss_col: ["sum", "mean"],
        cfg.escalated_col: "mean",
        cfg.control_failed_col: "mean",
    }

    # Build dynamic aggregation dict without missing columns
    dynamic_aggs = {}
    for k, v in aggs.items():
        if k in df.columns or v == "size":
            dynamic_aggs[k] = v

    g = df.groupby(cfg.client_col).agg(dynamic_aggs)
    # Flatten multi-index columns
    g.columns = ["_".join([c for c in col if c]) if isinstance(col, tuple) else col for col in g.columns]
    g = g.rename(columns={
        f"{cfg.event_id_col}_count": "event_count",
        "size": "event_count"
    })
    g["event_count"] = g.get("event_count", df.groupby(cfg.client_col).size())
    return g.reset_index()


# Trend Analysis


def monthly_trends(df: pd.DataFrame, cfg: RiskConfig) -> pd.DataFrame:
    group_cols = ["event_month"]
    metrics = {"event_count": ("size")}
    if cfg.loss_col in df.columns:
        metrics[cfg.loss_col] = "sum"
    if cfg.severity_col in df.columns:
        metrics[cfg.severity_col] = "mean"
    out = df.groupby(group_cols).agg(metrics)
    out.columns = ["event_count"] + [c for c in out.columns if c != "size"]
    return out.reset_index().sort_values("event_month")

def plot_monthly_trends(trends_df: pd.DataFrame, cfg: RiskConfig, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.lineplot(data=trends_df, x="event_month", y="event_count", ax=ax, marker="o")
    plt.xticks(rotation=45)
    ax.set_title("Monthly Risk Event Count")
    ax.set_xlabel("Month")
    ax.set_ylabel("Events")
    plt.tight_layout()
    fig.savefig(os.path.join(output_dir, "monthly_event_count.png"))
    plt.close(fig)


# Anomaly Detection


def detect_anomalies(df: pd.DataFrame, cfg: RiskConfig, features: Optional[List[str]] = None, contamination: float = 0.02) -> pd.DataFrame:
    """
    Uses IsolationForest to flag anomalous events based on selected features.
    """
    if features is None:
        features = [cfg.severity_col, cfg.loss_col, "ttr_days"]
        features = [f for f in features if f in df.columns]

    X = df[features].fillna(df[features].median())
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)

    iso = IsolationForest(n_estimators=300, contamination=contamination, random_state=42)
    df["anomaly_score"] = iso.fit_predict(Xs)  # -1 anomalous, 1 normal
    df["is_anomaly"] = (df["anomaly_score"] == -1).astype(int)
    return df


# Predictive Models


def poisson_event_count_model(df: pd.DataFrame, cfg: RiskConfig) -> pd.DataFrame:
    """
    Poisson regression to predict monthly event count per client.
    """
    # Prepare monthly client counts
    agg = df.groupby([cfg.client_col, "event_month"]).size().reset_index(name="events")
    # Simple features: last month count, rolling mean, control failure rate by client
    client_cf = df.groupby(cfg.client_col)[cfg.control_failed_col].mean() if cfg.control_failed_col in df.columns else None
    agg["month_idx"] = pd.to_datetime(agg["event_month"]).dt.to_period("M").astype(str)
    agg["month_idx"] = pd.to_datetime(agg["month_idx"])
    agg = agg.sort_values([cfg.client_col, "month_idx"])

    # Create lag feature per client
    agg["events_lag1"] = agg.groupby(cfg.client_col)["events"].shift(1).fillna(0)

    # Control failure feature
    if client_cf is not None:
        agg["client_cf_rate"] = agg[cfg.client_col].map(client_cf).fillna(0)
    else:
        agg["client_cf_rate"] = 0.0

    # Filter clients with enough history
    counts_per_client = agg.groupby(cfg.client_col).size()
    valid_clients = counts_per_client[counts_per_client >= cfg.min_events_per_client].index
    train = agg[agg[cfg.client_col].isin(valid_clients)].copy()

    if len(train) < 20:
        return pd.DataFrame({"note": ["Insufficient data for Poisson model"]})

    X = train[["events_lag1", "client_cf_rate"]]
    y = train["events"]

    model = PoissonRegressor(alpha=0.01, fit_intercept=True, max_iter=500)
    model.fit(X, y)

    train["pred_events"] = model.predict(X)
    mae = mean_absolute_error(y, train["pred_events"])

    # Coefficients
    coefs = pd.DataFrame({
        "feature": ["intercept", "events_lag1", "client_cf_rate"],
        "coefficient": [model.intercept_] + list(model.coef_)
    })

    summary = pd.DataFrame({"metric": ["MAE"], "value": [mae]})

    return {
        "predictions": train,
        "coefficients": coefs,
        "summary": summary
    }

def logistic_high_severity_model(df: pd.DataFrame, cfg: RiskConfig) -> Dict[str, pd.DataFrame]:
    """
    Logistic regression to predict whether an event is high severity (>= threshold).
    Features: monetary_loss, control_failed, escalated, ttr_days (if available).
    """
    if cfg.severity_col not in df.columns:
        return {"summary": pd.DataFrame({"note": ["Severity column missing"]})}

    dfm = df.copy()
    dfm["y_high_sev"] = (dfm[cfg.severity_col] >= cfg.high_severity_threshold).astype(int)

    features = []
    if cfg.loss_col in dfm.columns:
        features.append(cfg.loss_col)
    if cfg.control_failed_col in dfm.columns:
        features.append(cfg.control_failed_col)
    if cfg.escalated_col in dfm.columns:
        features.append(cfg.escalated_col)
    if "ttr_days" in dfm.columns:
        features.append("ttr_days")

    if len(features) == 0:
        return {"summary": pd.DataFrame({"note": ["No suitable features found"]})}

    X = dfm[features].fillna(dfm[features].median())
    y = dfm["y_high_sev"]

    # Train/test split by time (simple: last 20% as test)
    dfm = dfm.sort_values(cfg.date_col)
    split_idx = int(0.8 * len(dfm))
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_prob)
    prf = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)
    report = classification_report(y_test, y_pred, output_dict=True)

    coefs = pd.DataFrame({"feature": ["intercept"] + features,
                          "coefficient": [model.intercept_[0]] + list(model.coef_[0])})

    metrics = pd.DataFrame({
        "metric": ["ROC_AUC", "Precision", "Recall", "F1"],
        "value": [auc, prf[0], prf[1], prf[2]]
    })

    return {
        "coefficients": coefs,
        "metrics": metrics,
        "classification_report": pd.DataFrame(report).T
    }


# Survival Analysis (Kaplan–Meier)


def kaplan_meier(df: pd.DataFrame, duration_col: str = "ttr_days", event_observed_col: Optional[str] = None) -> pd.DataFrame:
    """
    Simple Kaplan–Meier estimator for survival function S(t).
    duration_col: time to resolution in days
    event_observed_col: 1 if resolved (event occurred), 0 if censored (still open)
                        If None, we infer: resolved if duration not na and duration >=0
    Returns: DataFrame with columns: time, n_at_risk, n_events, survival
    """
    d = df.copy()
    d = d[~d[duration_col].isna()].copy()
    d[duration_col] = d[duration_col].astype(int)

    if event_observed_col is None:
        # Assume event observed if duration >= 0 and status is Closed
        event_observed = np.ones(len(d))
        if "status" in d.columns:
            event_observed = (d["status"].str.lower() == "closed").astype(int)
        d["event_observed"] = event_observed
    else:
        d["event_observed"] = d[event_observed_col].astype(int)

    # Times in ascending order
    times = np.sort(d[duration_col].unique())
    n = len(d)

    survival = []
    n_at_risk_prev = n
    s_prev = 1.0

    for t in times:
        # At risk: entities with duration >= t
        n_at_risk = ((d[duration_col] >= t)).sum()
        # Events at time t (resolved exactly at t)
        n_events = ((d[duration_col] == t) & (d["event_observed"] == 1)).sum()

        # KM step: S(t) = S(t-1) * (1 - d_i / n_i)
        s_t = s_prev * (1 - (n_events / max(n_at_risk, 1)))
        survival.append({"time": t, "n_at_risk": int(n_at_risk), "n_events": int(n_events), "survival": float(s_t)})
        s_prev = s_t
        n_at_risk_prev = n_at_risk

    return pd.DataFrame(survival)

def plot_km(km_df: pd.DataFrame, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.step(km_df["time"], km_df["survival"], where="post")
    ax.set_title("Kaplan–Meier Survival: Probability of NOT Resolved by Day t")
    ax.set_xlabel("Days since event")
    ax.set_ylabel("Survival S(t)")
    plt.tight_layout()
    fig.savefig(os.path.join(output_dir, "km_survival.png"))
    plt.close(fig)


# Reporting

def write_report(kri: Dict[str, float],
                 client_metrics: pd.DataFrame,
                 trends: pd.DataFrame,
                 anomalies_df: pd.DataFrame,
                 poisson_outputs,
                 logistic_outputs,
                 km_df: pd.DataFrame,
                 cfg: RiskConfig):
    os.makedirs(cfg.output_dir, exist_ok=True)
    report_path = os.path.join(cfg.output_dir, cfg.report_xlsx)

    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        pd.DataFrame([kri]).to_excel(writer, sheet_name="KRI_Summary", index=False)
        client_metrics.to_excel(writer, sheet_name="Client_Metrics", index=False)
        trends.to_excel(writer, sheet_name="Monthly_Trends", index=False)
        anomalies_df[[cfg.client_col, cfg.event_id_col if cfg.event_id_col in anomalies_df.columns else cfg.date_col,
                      "is_anomaly", cfg.severity_col if cfg.severity_col in anomalies_df.columns else None,
                      cfg.loss_col if cfg.loss_col in anomalies_df.columns else None]].dropna(axis=1, how="all") \
            .to_excel(writer, sheet_name="Anomalies", index=False)

        # Poisson model outputs
        if isinstance(poisson_outputs, dict):
            poisson_outputs["predictions"].to_excel(writer, sheet_name="Poisson_Predictions", index=False)
            poisson_outputs["coefficients"].to_excel(writer, sheet_name="Poisson_Coefficients", index=False)
            poisson_outputs["summary"].to_excel(writer, sheet_name="Poisson_Summary", index=False)
        else:
            pd.DataFrame(poisson_outputs).to_excel(writer, sheet_name="Poisson_Summary", index=False)

        # Logistic model outputs
        if isinstance(logistic_outputs, dict):
            logistic_outputs["coefficients"].to_excel(writer, sheet_name="Logistic_Coefficients", index=False)
            logistic_outputs["metrics"].to_excel(writer, sheet_name="Logistic_Metrics", index=False)
            logistic_outputs["classification_report"].to_excel(writer, sheet_name="Logistic_Report", index=True)
        else:
            pd.DataFrame(logistic_outputs).to_excel(writer, sheet_name="Logistic_Summary", index=False)

        # Survival
        km_df.to_excel(writer, sheet_name="KM_Survival", index=False)

    # Save plots
    plots_dir = os.path.join(cfg.output_dir, cfg.plots_dir)
    os.makedirs(plots_dir, exist_ok=True)
    # Monthly trend plot
    try:
        plot_monthly_trends(trends, cfg, plots_dir)
    except Exception as e:
        print(f"[WARN] Could not create monthly trend plot: {e}")
    # KM plot
    try:
        plot_km(km_df, plots_dir)
    except Exception as e:
        print(f"[WARN] Could not create KM plot: {e}")

    print(f"✅ Report written to: {report_path}")
    print(f"📊 Plots saved under: {plots_dir}")


# Pipeline

def run_pipeline(input_path: str, cfg: Optional[RiskConfig] = None):
    cfg = cfg or RiskConfig()

    # 1) Load
    df = load_data(input_path)

    # 2) Clean & Feature Engineering
    df = clean_data(df, cfg)

    # 3) KRIs
    kri = compute_kri(df, cfg)
    print("KRI Summary:", kri)

    # 4) Client-level metrics
    cl_metrics = client_level_metrics(df, cfg)

    # 5) Monthly trends
    trends = monthly_trends(df, cfg)

    # 6) Anomaly detection
    df_anom = detect_anomalies(df, cfg)

    # 7) Predictive modeling
    poisson_out = poisson_event_count_model(df, cfg)
    logistic_out = logistic_high_severity_model(df, cfg)

    # 8) Survival (TTR)
    km_df = kaplan_meier(df, duration_col="ttr_days")

    # 9) Report
    write_report(kri, cl_metrics, trends, df_anom, poisson_out, logistic_out, km_df, cfg)

